In [2]:
# ============================================================
# 07 - Demand Forecasting
# Load the hourly demand data created in Notebook 06
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\arudk\Downloads\UrbanFlow_AI")

hourly_path = (
    BASE_DIR
    / "data"
    / "processed"
    / "taxi_zone"
    / "hourly_demand.parquet"
)

pickup_path = (
    BASE_DIR
    / "data"
    / "processed"
    / "taxi_zone"
    / "pickup_zone_statistics.csv"
)

print("Loading hourly demand data...")
hourly = pd.read_parquet(hourly_path)

print("Hourly demand shape:", hourly.shape)
print("\nColumns:")
print(hourly.columns.tolist())

print("\nFirst 5 rows:")
display(hourly.head())

Loading hourly demand data...
Hourly demand shape: (8766, 3)

Columns:
['pickup_date', 'pickup_hour', 'trip_count']

First 5 rows:


,pickup_date,pickup_hour,trip_count
0,2008-12-31,23,2
1,2009-01-01,0,3
2,2009-01-01,8,1
3,2009-01-01,12,1
4,2009-01-01,14,1


In [4]:
# ============================================================
# Quick timestamp sanity check
# ============================================================

print("Date range:")
print("Minimum:", hourly["pickup_date"].min())
print("Maximum:", hourly["pickup_date"].max())

print("\nDate dtype:")
print(hourly["pickup_date"].dtype)

print("\nUnique dates:", hourly["pickup_date"].nunique())
print("Rows:", len(hourly))

print("\nLast 5 rows:")
display(hourly.tail())

print("\nRows around expected 2025-2026 period:")
display(
    hourly[
        (hourly["pickup_date"] >= "2025-04-01") &
        (hourly["pickup_date"] <= "2026-03-31")
    ].head(20)
)

Date range:
Minimum: 2008-12-31 00:00:00
Maximum: 2026-04-01 00:00:00

Date dtype:
datetime64[ns]

Unique dates: 369
Rows: 8766

Last 5 rows:


,pickup_date,pickup_hour,trip_count
8761,2026-03-31,20,6745
8762,2026-03-31,21,7457
8763,2026-03-31,22,5855
8764,2026-03-31,23,3537
8765,2026-04-01,0,2



Rows around expected 2025-2026 period:


,pickup_date,pickup_hour,trip_count
6,2025-04-01,0,2666
7,2025-04-01,1,1376
8,2025-04-01,2,907
9,2025-04-01,3,375
10,2025-04-01,4,429
11,2025-04-01,5,824
12,2025-04-01,6,2156
13,2025-04-01,7,5103
14,2025-04-01,8,6776
15,2025-04-01,9,6503


In [6]:
# ============================================================
# Clean the hourly demand timeline
# ============================================================

# Keep only the actual competition period
hourly = hourly[
    (hourly["pickup_date"] >= "2025-04-01") &
    (hourly["pickup_date"] <= "2026-03-31")
].copy()

# Create one proper hourly timestamp
hourly["timestamp"] = (
    hourly["pickup_date"]
    + pd.to_timedelta(hourly["pickup_hour"], unit="h")
)

# Sort chronologically
hourly = hourly.sort_values("timestamp").reset_index(drop=True)

print("Cleaned hourly demand shape:", hourly.shape)
print("Start:", hourly["timestamp"].min())
print("End:", hourly["timestamp"].max())

print("\nTotal trips represented:", hourly["trip_count"].sum())

print("\nFirst 5 rows:")
display(hourly.head())

print("\nLast 5 rows:")
display(hourly.tail())

Cleaned hourly demand shape: (8759, 4)
Start: 2025-04-01 00:00:00
End: 2026-03-31 23:00:00

Total trips represented: 48601768

First 5 rows:


,pickup_date,pickup_hour,trip_count,timestamp
0,2025-04-01,0,2666,2025-04-01 00:00:00
1,2025-04-01,1,1376,2025-04-01 01:00:00
2,2025-04-01,2,907,2025-04-01 02:00:00
3,2025-04-01,3,375,2025-04-01 03:00:00
4,2025-04-01,4,429,2025-04-01 04:00:00



Last 5 rows:


,pickup_date,pickup_hour,trip_count,timestamp
8754,2026-03-31,19,7067,2026-03-31 19:00:00
8755,2026-03-31,20,6745,2026-03-31 20:00:00
8756,2026-03-31,21,7457,2026-03-31 21:00:00
8757,2026-03-31,22,5855,2026-03-31 22:00:00
8758,2026-03-31,23,3537,2026-03-31 23:00:00


In [8]:
# ============================================================
# Build forecasting features
#
# We use previous demand values to let the model learn:
# - very recent demand
# - same hour yesterday
# - same hour last week
# - rolling demand patterns
# ============================================================

# Make sure the data is in chronological order
hourly = hourly.sort_values("timestamp").reset_index(drop=True)

# Calendar features
hourly["hour"] = hourly["timestamp"].dt.hour
hourly["day_of_week"] = hourly["timestamp"].dt.dayofweek
hourly["day_of_month"] = hourly["timestamp"].dt.day
hourly["month"] = hourly["timestamp"].dt.month
hourly["is_weekend"] = (hourly["day_of_week"] >= 5).astype(int)

# Demand lag features
hourly["lag_1h"] = hourly["trip_count"].shift(1)
hourly["lag_2h"] = hourly["trip_count"].shift(2)
hourly["lag_3h"] = hourly["trip_count"].shift(3)

# Same hour on the previous day
hourly["lag_24h"] = hourly["trip_count"].shift(24)

# Same hour one week earlier
hourly["lag_168h"] = hourly["trip_count"].shift(168)

# Rolling demand statistics.
# shift(1) ensures that the current target is never included.
hourly["rolling_mean_24h"] = (
    hourly["trip_count"]
    .shift(1)
    .rolling(24)
    .mean()
)

hourly["rolling_mean_168h"] = (
    hourly["trip_count"]
    .shift(1)
    .rolling(168)
    .mean()
)

# Remove rows where lag/rolling features are unavailable
model_data = hourly.dropna().copy()

print("Original hourly rows:", len(hourly))
print("Modeling rows:", len(model_data))

print("\nFeature columns:")
print(model_data.columns.tolist())

print("\nDate range:")
print(model_data["timestamp"].min(), "to", model_data["timestamp"].max())

print("\nSample:")
display(model_data.head())

Original hourly rows: 8759
Modeling rows: 8591

Feature columns:
['pickup_date', 'pickup_hour', 'trip_count', 'timestamp', 'hour', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h']

Date range:
2025-04-08 00:00:00 to 2026-03-31 23:00:00

Sample:


,pickup_date,pickup_hour,trip_count,timestamp,hour,day_of_week,day_of_month,month,is_weekend,lag_1h,lag_2h,lag_3h,lag_24h,lag_168h,rolling_mean_24h,rolling_mean_168h
168,2025-04-08,0,1446,2025-04-08 00:00:00,0,1,8,4,0,2744.0,4616.0,6191.0,1577.0,2666.0,4826.208333,5546.607143
169,2025-04-08,1,577,2025-04-08 01:00:00,1,1,8,4,0,1446.0,2744.0,4616.0,729.0,1376.0,4820.750000,5539.345238
170,2025-04-08,2,340,2025-04-08 02:00:00,2,1,8,4,0,577.0,1446.0,2744.0,412.0,907.0,4814.416667,5534.589286
171,2025-04-08,3,237,2025-04-08 03:00:00,3,1,8,4,0,340.0,577.0,1446.0,316.0,375.0,4811.416667,5531.214286
172,2025-04-08,4,352,2025-04-08 04:00:00,4,1,8,4,0,237.0,340.0,577.0,479.0,429.0,4808.125000,5530.392857


In [10]:
# ============================================================
# Demand Forecasting Model
# Chronological train / validation / test split
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# Features available before the prediction hour
features = [
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",
    "lag_1h",
    "lag_2h",
    "lag_3h",
    "lag_24h",
    "lag_168h",
    "rolling_mean_24h",
    "rolling_mean_168h"
]

target = "trip_count"

X = model_data[features].copy()
y = model_data[target].copy()

# ------------------------------------------------------------
# Chronological split
# ------------------------------------------------------------

n = len(model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("TRAIN")
print(
    model_data.iloc[0]["timestamp"],
    "to",
    model_data.iloc[train_end - 1]["timestamp"]
)
print("Rows:", len(X_train))

print("\nVALIDATION")
print(
    model_data.iloc[train_end]["timestamp"],
    "to",
    model_data.iloc[val_end - 1]["timestamp"]
)
print("Rows:", len(X_val))

print("\nTEST")
print(
    model_data.iloc[val_end]["timestamp"],
    "to",
    model_data.iloc[-1]["timestamp"]
)
print("Rows:", len(X_test))


# ------------------------------------------------------------
# Try LightGBM first
# ------------------------------------------------------------

try:
    from lightgbm import LGBMRegressor

    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1
    )

    model_name = "LightGBM"

except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor

    model = HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    model_name = "HistGradientBoosting"


print("\nTraining model:", model_name)

model.fit(X_train, y_train)

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

def evaluate_model(model, X_data, y_data, name):
    predictions = model.predict(X_data)

    mae = mean_absolute_error(y_data, predictions)
    rmse = np.sqrt(mean_squared_error(y_data, predictions))
    r2 = r2_score(y_data, predictions)

    print(f"\n{name}")
    print(f"MAE : {mae:.2f} trips/hour")
    print(f"RMSE: {rmse:.2f} trips/hour")
    print(f"R²  : {r2:.4f}")

    return predictions, mae, rmse, r2


val_pred, val_mae, val_rmse, val_r2 = evaluate_model(
    model, X_val, y_val, "Validation Results"
)

test_pred, test_mae, test_rmse, test_r2 = evaluate_model(
    model, X_test, y_test, "Test Results"
)

TRAIN
2025-04-08 00:00:00 to 2025-12-14 12:00:00
Rows: 6013

VALIDATION
2025-12-14 13:00:00 to 2026-02-06 05:00:00
Rows: 1289

TEST
2026-02-06 06:00:00 to 2026-03-31 23:00:00
Rows: 1289

Training model: HistGradientBoosting


C:\Users\arudk\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\arudk\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro


Validation Results
MAE : 366.29 trips/hour
RMSE: 566.83 trips/hour
R²  : 0.9622

Test Results
MAE : 373.07 trips/hour
RMSE: 565.39 trips/hour
R²  : 0.9625


In [12]:
# ============================================================
# Build zone-level hourly demand for the most important zones
# ============================================================

from collections import defaultdict

taxi_dir = BASE_DIR / "data" / "raw" / "taxi"

# Top 10 pickup zones from Notebook 06
TOP_ZONES = [
    237, 132, 161, 236, 186,
    162, 230, 142, 170, 234
]

print("Top zones selected for forecasting:")
print(TOP_ZONES)

# Store hourly counts for each zone
zone_hourly = defaultdict(int)

taxi_files = sorted(taxi_dir.glob("*.csv"))

print("\nTaxi files found:", len(taxi_files))

for file_path in taxi_files:

    print("Processing:", file_path.name)

    for chunk in pd.read_csv(
        file_path,
        usecols=["pickup_timestamp", "origin_loc_id"],
        chunksize=500_000
    ):

        chunk["pickup_timestamp"] = pd.to_datetime(
            chunk["pickup_timestamp"],
            errors="coerce"
        )

        # Keep only the selected top zones
        chunk = chunk[
            chunk["origin_loc_id"].isin(TOP_ZONES)
        ].copy()

        if chunk.empty:
            continue

        # Convert timestamp to hourly timestamp
        chunk["timestamp"] = chunk["pickup_timestamp"].dt.floor("h")

        counts = (
            chunk
            .groupby(["timestamp", "origin_loc_id"])
            .size()
        )

        for (timestamp, zone), count in counts.items():
            zone_hourly[(timestamp, zone)] += count

print("\nAggregation complete.")
print("Zone-hour combinations:", len(zone_hourly))

Top zones selected for forecasting:
[237, 132, 161, 236, 186, 162, 230, 142, 170, 234]

Taxi files found: 12
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

Aggregation complete.
Zone-hour combinations: 87459


In [14]:
# ============================================================
# Create a complete hourly time series for the top 10 zones
# ============================================================

# Convert the dictionary into a DataFrame
zone_hourly_df = pd.DataFrame(
    [
        (timestamp, zone, count)
        for (timestamp, zone), count in zone_hourly.items()
    ],
    columns=["timestamp", "zone_id", "trip_count"]
)

zone_hourly_df["timestamp"] = pd.to_datetime(zone_hourly_df["timestamp"])

# Complete hourly timeline
full_hours = pd.date_range(
    start="2025-04-01 00:00:00",
    end="2026-03-31 23:00:00",
    freq="h"
)

# Every hour × every top zone
complete_index = pd.MultiIndex.from_product(
    [full_hours, TOP_ZONES],
    names=["timestamp", "zone_id"]
)

# Fill absent zone-hour combinations with zero demand
zone_hourly_df = (
    zone_hourly_df
    .set_index(["timestamp", "zone_id"])
    .reindex(complete_index, fill_value=0)
    .reset_index()
)

# Calendar features
zone_hourly_df["hour"] = zone_hourly_df["timestamp"].dt.hour
zone_hourly_df["day_of_week"] = zone_hourly_df["timestamp"].dt.dayofweek
zone_hourly_df["day_of_month"] = zone_hourly_df["timestamp"].dt.day
zone_hourly_df["month"] = zone_hourly_df["timestamp"].dt.month
zone_hourly_df["is_weekend"] = (
    zone_hourly_df["day_of_week"] >= 5
).astype(int)

# Sort before creating lags
zone_hourly_df = zone_hourly_df.sort_values(
    ["zone_id", "timestamp"]
).reset_index(drop=True)

# Zone-specific demand lags
zone_hourly_df["lag_1h"] = (
    zone_hourly_df.groupby("zone_id")["trip_count"].shift(1)
)

zone_hourly_df["lag_24h"] = (
    zone_hourly_df.groupby("zone_id")["trip_count"].shift(24)
)

zone_hourly_df["lag_168h"] = (
    zone_hourly_df.groupby("zone_id")["trip_count"].shift(168)
)

# Rolling features using only previous observations
zone_hourly_df["rolling_mean_24h"] = (
    zone_hourly_df
    .groupby("zone_id")["trip_count"]
    .transform(lambda x: x.shift(1).rolling(24).mean())
)

zone_hourly_df["rolling_mean_168h"] = (
    zone_hourly_df
    .groupby("zone_id")["trip_count"]
    .transform(lambda x: x.shift(1).rolling(168).mean())
)

# Remove rows where the longest lag is unavailable
zone_model_data = zone_hourly_df.dropna().copy()

print("Complete zone-hour rows:", len(zone_hourly_df))
print("Modeling rows:", len(zone_model_data))

print("\nZones:")
print(sorted(zone_model_data["zone_id"].unique()))

print("\nDemand range:")
print(
    zone_model_data["trip_count"].min(),
    "to",
    zone_model_data["trip_count"].max()
)

print("\nSample:")
display(zone_model_data.head(10))

Complete zone-hour rows: 87600
Modeling rows: 85920

Zones:
[132, 142, 161, 162, 170, 186, 230, 234, 236, 237]

Demand range:
0 to 945

Sample:


,timestamp,zone_id,trip_count,hour,day_of_week,day_of_month,month,is_weekend,lag_1h,lag_24h,lag_168h,rolling_mean_24h,rolling_mean_168h
168,2025-04-08 00:00:00,132,193,0,1,8,4,0,339.0,212.0,200.0,253.916667,219.184524
169,2025-04-08 01:00:00,132,80,1,1,8,4,0,193.0,109.0,317.0,253.125000,219.142857
170,2025-04-08 02:00:00,132,46,2,1,8,4,0,80.0,39.0,231.0,251.916667,217.732143
171,2025-04-08 03:00:00,132,13,3,1,8,4,0,46.0,14.0,80.0,252.208333,216.630952
172,2025-04-08 04:00:00,132,33,4,1,8,4,0,13.0,19.0,58.0,252.166667,216.232143
173,2025-04-08 05:00:00,132,59,5,1,8,4,0,33.0,85.0,30.0,252.750000,216.083333
174,2025-04-08 06:00:00,132,133,6,1,8,4,0,59.0,195.0,177.0,251.666667,216.255952
175,2025-04-08 07:00:00,132,158,7,1,8,4,0,133.0,178.0,213.0,249.083333,215.994048
176,2025-04-08 08:00:00,132,69,8,1,8,4,0,158.0,127.0,93.0,248.250000,215.666667
177,2025-04-08 09:00:00,132,88,9,1,8,4,0,69.0,178.0,164.0,245.833333,215.523810


In [16]:
# ============================================================
# Top-Zone Demand Forecasting Model
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Features known when making the forecast
zone_features = [
    "zone_id",
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "rolling_mean_24h",
    "rolling_mean_168h"
]

target = "trip_count"

X = zone_model_data[zone_features].copy()
y = zone_model_data[target].copy()

# ------------------------------------------------------------
# Chronological split
# ------------------------------------------------------------

n = len(zone_model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("TRAIN")
print(
    zone_model_data.iloc[0]["timestamp"],
    "to",
    zone_model_data.iloc[train_end - 1]["timestamp"]
)
print("Rows:", len(X_train))

print("\nVALIDATION")
print(
    zone_model_data.iloc[train_end]["timestamp"],
    "to",
    zone_model_data.iloc[val_end - 1]["timestamp"]
)
print("Rows:", len(X_val))

print("\nTEST")
print(
    zone_model_data.iloc[val_end]["timestamp"],
    "to",
    zone_model_data.iloc[-1]["timestamp"]
)
print("Rows:", len(X_test))


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("\nTraining top-zone demand model...")

zone_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

zone_model.fit(X_train, y_train)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

def evaluate_zone_model(model, X_data, y_data, name):

    predictions = model.predict(X_data)

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    mae = mean_absolute_error(y_data, predictions)
    rmse = np.sqrt(mean_squared_error(y_data, predictions))
    r2 = r2_score(y_data, predictions)

    print(f"\n{name}")
    print(f"MAE : {mae:.2f} trips/hour")
    print(f"RMSE: {rmse:.2f} trips/hour")
    print(f"R²  : {r2:.4f}")

    return predictions, mae, rmse, r2


val_pred, val_mae, val_rmse, val_r2 = evaluate_zone_model(
    zone_model,
    X_val,
    y_val,
    "Validation Results"
)

test_pred, test_mae, test_rmse, test_r2 = evaluate_zone_model(
    zone_model,
    X_test,
    y_test,
    "Test Results"
)

TRAIN
2025-04-08 00:00:00 to 2026-03-31 22:00:00
Rows: 60143

VALIDATION
2026-03-31 23:00:00 to 2025-10-03 23:00:00
Rows: 12889

TEST
2025-10-04 00:00:00 to 2026-03-31 23:00:00
Rows: 12888

Training top-zone demand model...

Validation Results
MAE : 20.82 trips/hour
RMSE: 31.80 trips/hour
R²  : 0.9423

Test Results
MAE : 29.92 trips/hour
RMSE: 44.20 trips/hour
R²  : 0.9435


In [18]:
# ============================================================
# Correct chronological split for top-zone demand forecasting
# ============================================================

# IMPORTANT:
# The previous dataset was ordered by zone.
# For forecasting, all zones must be ordered by time first.

zone_model_data = (
    zone_model_data
    .sort_values(["timestamp", "zone_id"])
    .reset_index(drop=True)
)

print("Overall time range:")
print(
    zone_model_data["timestamp"].min(),
    "to",
    zone_model_data["timestamp"].max()
)

# ------------------------------------------------------------
# Features and target
# ------------------------------------------------------------

zone_features = [
    "zone_id",
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "rolling_mean_24h",
    "rolling_mean_168h"
]

target = "trip_count"

X = zone_model_data[zone_features].copy()
y = zone_model_data[target].copy()

# ------------------------------------------------------------
# Chronological split
# ------------------------------------------------------------

n = len(zone_model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("\nTRAIN")
print(
    zone_model_data.iloc[0]["timestamp"],
    "to",
    zone_model_data.iloc[train_end - 1]["timestamp"]
)
print("Rows:", len(X_train))

print("\nVALIDATION")
print(
    zone_model_data.iloc[train_end]["timestamp"],
    "to",
    zone_model_data.iloc[val_end - 1]["timestamp"]
)
print("Rows:", len(X_val))

print("\nTEST")
print(
    zone_model_data.iloc[val_end]["timestamp"],
    "to",
    zone_model_data.iloc[-1]["timestamp"]
)
print("Rows:", len(X_test))

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("\nTraining corrected top-zone demand model...")

zone_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

zone_model.fit(X_train, y_train)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

def evaluate_zone_model(model, X_data, y_data, name):

    predictions = model.predict(X_data)

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    mae = mean_absolute_error(y_data, predictions)
    rmse = np.sqrt(mean_squared_error(y_data, predictions))
    r2 = r2_score(y_data, predictions)

    print(f"\n{name}")
    print(f"MAE : {mae:.2f} trips/hour")
    print(f"RMSE: {rmse:.2f} trips/hour")
    print(f"R²  : {r2:.4f}")

    return predictions, mae, rmse, r2


val_pred, val_mae, val_rmse, val_r2 = evaluate_zone_model(
    zone_model,
    X_val,
    y_val,
    "Validation Results"
)

test_pred, test_mae, test_rmse, test_r2 = evaluate_zone_model(
    zone_model,
    X_test,
    y_test,
    "Test Results"
)

Overall time range:
2025-04-08 00:00:00 to 2026-03-31 23:00:00

TRAIN
2025-04-08 00:00:00 to 2025-12-14 14:00:00
Rows: 60143

VALIDATION
2025-12-14 14:00:00 to 2026-02-06 07:00:00
Rows: 12889

TEST
2026-02-06 07:00:00 to 2026-03-31 23:00:00
Rows: 12888

Training corrected top-zone demand model...

Validation Results
MAE : 23.55 trips/hour
RMSE: 35.34 trips/hour
R²  : 0.9331

Test Results
MAE : 22.51 trips/hour
RMSE: 34.04 trips/hour
R²  : 0.9394


## 4. Model Evaluation and Artifact Saving

A gradient-boosting regression model is used to forecast hourly pickup demand across the ten highest-volume taxi zones.

The data is divided chronologically into training, validation, and test periods to preserve the temporal structure of the forecasting problem and prevent future observations from influencing model development.

Model performance is evaluated using MAE, RMSE, and R². The final model and evaluation metrics are saved as reproducible project artifacts for later integration into the technical report and final submission.

In [21]:
# ============================================================
# Save model artifacts and evaluation results
# ============================================================

MODEL_DIR = BASE_DIR / "models" / "demand"
OUTPUT_DIR = BASE_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Persist the trained model
model_path = MODEL_DIR / "top_zone_demand_model.pkl"
joblib.dump(zone_model, model_path)

# Store the evaluation results in a compact report-ready table
demand_results = pd.DataFrame({
    "model": ["HistGradientBoostingRegressor"],
    "target": ["Hourly pickup demand"],
    "zones": [len(TOP_ZONES)],
    "validation_mae": [val_mae],
    "validation_rmse": [val_rmse],
    "validation_r2": [val_r2],
    "test_mae": [test_mae],
    "test_rmse": [test_rmse],
    "test_r2": [test_r2]
})

results_path = OUTPUT_DIR / "demand_model_results.csv"
demand_results.to_csv(results_path, index=False)

print("Model saved to:")
print(model_path)

print("\nEvaluation results saved to:")
print(results_path)

print("\nFinal test performance:")
display(demand_results)

Model saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\models\demand\top_zone_demand_model.pkl

Evaluation results saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\demand_model_results.csv

Final test performance:


,model,target,zones,validation_mae,validation_rmse,validation_r2,test_mae,test_rmse,test_r2
0,HistGradientBoostingRegressor,Hourly pickup demand,10,23.546835,35.341348,0.93312,22.514409,34.041943,0.939418


## 5. Short-Horizon Demand Forecasting

The trained model is used to generate forward demand forecasts for the ten highest-volume pickup zones.

Two operational horizons are produced:

- **24-hour forecast** for near-term fleet and driver allocation.
- **72-hour forecast** for short-term planning and resource scheduling.

Forecasts are generated recursively. For each future hour, the model uses the most recently available demand history and previously generated predictions to construct the required lag features.

This approach prevents future actual demand from being used during forecasting.

In [24]:
# ============================================================
# Generate recursive 24-hour and 72-hour forecasts
# ============================================================

# We start from the final observed hour in the historical dataset.
last_timestamp = zone_hourly_df["timestamp"].max()

print("Last observed timestamp:", last_timestamp)

# Keep the most recent 168 hours for each zone.
# This is sufficient to construct the weekly lag and rolling features.
history = (
    zone_hourly_df[
        zone_hourly_df["timestamp"] <= last_timestamp
    ][["timestamp", "zone_id", "trip_count"]]
    .copy()
)

history = history.sort_values(
    ["zone_id", "timestamp"]
).reset_index(drop=True)

history = (
    history
    .groupby("zone_id", group_keys=False)
    .tail(168)
    .copy()
)

print("Historical rows retained for forecasting:", len(history))


def build_forecast_features(history_df, timestamp, zone_id):
    """
    Construct the feature vector required for one future
    zone-hour using only information available up to that point.
    """

    zone_history = (
        history_df[history_df["zone_id"] == zone_id]
        .sort_values("timestamp")
    )

    demand = zone_history["trip_count"].to_numpy()

    # Recent lags
    lag_1h = demand[-1]
    lag_24h = demand[-24]
    lag_168h = demand[-168]

    # Rolling windows use historical values only
    rolling_mean_24h = np.mean(demand[-24:])
    rolling_mean_168h = np.mean(demand[-168:])

    return {
        "zone_id": zone_id,
        "hour": timestamp.hour,
        "day_of_week": timestamp.dayofweek,
        "day_of_month": timestamp.day,
        "month": timestamp.month,
        "is_weekend": int(timestamp.dayofweek >= 5),
        "lag_1h": lag_1h,
        "lag_24h": lag_24h,
        "lag_168h": lag_168h,
        "rolling_mean_24h": rolling_mean_24h,
        "rolling_mean_168h": rolling_mean_168h
    }


def recursive_forecast(history_df, hours_ahead):
    """
    Generate a recursive multi-step forecast for all top zones.
    """

    working_history = history_df.copy()
    forecasts = []

    future_timestamps = pd.date_range(
        start=last_timestamp + pd.Timedelta(hours=1),
        periods=hours_ahead,
        freq="h"
    )

    for timestamp in future_timestamps:

        for zone_id in TOP_ZONES:

            features = build_forecast_features(
                working_history,
                timestamp,
                zone_id
            )

            X_future = pd.DataFrame(
                [features],
                columns=zone_features
            )

            prediction = zone_model.predict(X_future)[0]

            # Demand cannot be negative
            prediction = max(0, prediction)

            forecasts.append({
                "timestamp": timestamp,
                "zone_id": zone_id,
                "predicted_demand": prediction
            })

            # Feed the prediction back into the history so that
            # later forecast steps can use it as a lagged value.
            working_history = pd.concat(
                [
                    working_history,
                    pd.DataFrame({
                        "timestamp": [timestamp],
                        "zone_id": [zone_id],
                        "trip_count": [prediction]
                    })
                ],
                ignore_index=True
            )

    return pd.DataFrame(forecasts)


# Generate both operational horizons
forecast_24h = recursive_forecast(history, 24)
forecast_72h = recursive_forecast(history, 72)

print("\n24-hour forecast rows:", len(forecast_24h))
print("72-hour forecast rows:", len(forecast_72h))

print("\n24-hour forecast sample:")
display(forecast_24h.head(10))

print("\n72-hour forecast sample:")
display(forecast_72h.head(10))

Last observed timestamp: 2026-03-31 23:00:00
Historical rows retained for forecasting: 1680

24-hour forecast rows: 240
72-hour forecast rows: 720

24-hour forecast sample:


,timestamp,zone_id,predicted_demand
0,2026-04-01,237,38.579603
1,2026-04-01,132,249.661915
2,2026-04-01,161,66.122847
3,2026-04-01,236,19.557827
4,2026-04-01,186,54.986371
5,2026-04-01,162,47.001163
6,2026-04-01,230,106.690377
7,2026-04-01,142,36.905599
8,2026-04-01,170,42.735508
9,2026-04-01,234,60.335360



72-hour forecast sample:


,timestamp,zone_id,predicted_demand
0,2026-04-01,237,38.579603
1,2026-04-01,132,249.661915
2,2026-04-01,161,66.122847
3,2026-04-01,236,19.557827
4,2026-04-01,186,54.986371
5,2026-04-01,162,47.001163
6,2026-04-01,230,106.690377
7,2026-04-01,142,36.905599
8,2026-04-01,170,42.735508
9,2026-04-01,234,60.335360


## 6. Forecast Outputs and Zone-Level Interpretation

The recursive forecasting outputs are enriched with the corresponding taxi-zone metadata so that model predictions can be interpreted in operational terms.

Both the 24-hour and 72-hour forecasts are retained. A zone-level summary is also generated to identify areas expected to experience the highest demand during the forecast horizon.

These outputs provide a direct basis for fleet allocation, driver positioning, and short-term capacity planning.

In [27]:
# ============================================================
# Enrich forecasts with taxi-zone metadata
# ============================================================

zone_path = BASE_DIR / "data" / "raw" / "zone" / "Urban_Flow_Analytics_Zone_Dataset.csv"

zones = pd.read_csv(zone_path)

zone_lookup = zones[
    ["loc_id", "borough_name", "zone_name", "service_zone"]
].rename(
    columns={"loc_id": "zone_id"}
)

# Attach human-readable zone information
forecast_24h = forecast_24h.merge(
    zone_lookup,
    on="zone_id",
    how="left"
)

forecast_72h = forecast_72h.merge(
    zone_lookup,
    on="zone_id",
    how="left"
)

# ------------------------------------------------------------
# Save forecast outputs
# ------------------------------------------------------------

forecast_24h_path = OUTPUT_DIR / "demand_forecast_24h.csv"
forecast_72h_path = OUTPUT_DIR / "demand_forecast_72h.csv"

forecast_24h.to_csv(forecast_24h_path, index=False)
forecast_72h.to_csv(forecast_72h_path, index=False)

print("24-hour forecast saved to:")
print(forecast_24h_path)

print("\n72-hour forecast saved to:")
print(forecast_72h_path)

# ------------------------------------------------------------
# Zone-level summary
# ------------------------------------------------------------

zone_summary = (
    forecast_72h
    .groupby(
        ["zone_id", "borough_name", "zone_name"],
        as_index=False
    )["predicted_demand"]
    .agg(
        forecast_total="sum",
        forecast_mean="mean",
        forecast_peak="max"
    )
    .sort_values("forecast_total", ascending=False)
)

print("\nTop zones by predicted 72-hour demand:")
display(zone_summary)

24-hour forecast saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\demand_forecast_24h.csv

72-hour forecast saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\demand_forecast_72h.csv

Top zones by predicted 72-hour demand:


,zone_id,borough_name,zone_name,forecast_total,forecast_mean,forecast_peak
2,161,Manhattan,Midtown Center,18662.737250,259.204684,625.079708
9,237,Manhattan,Upper East Side South,16958.767949,235.538444,467.607543
0,132,Queens,JFK Airport,15411.043469,214.042270,398.093773
8,236,Manhattan,Upper East Side North,14246.923291,197.873935,368.889289
3,162,Manhattan,Midtown East,13458.723227,186.926711,427.169351
6,230,Manhattan,Times Sq/Theatre District,13279.601811,184.438914,547.802145
5,186,Manhattan,Penn Station/Madison Sq West,12603.459612,175.048050,281.161392
4,170,Manhattan,Murray Hill,11191.282872,155.434484,328.524842
1,142,Manhattan,Lincoln Square East,11177.806709,155.247315,364.948181
7,234,Manhattan,Union Sq,10713.251288,148.795157,331.667191
